In [ ]:
# ============================================================
# 1. Mount Drive & Set Paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ✅ CHANGE THIS to the folder where your model files are stored
model_folder = '/content/drive/MyDrive/model/'

# ============================================================
# 2. Install & Import Libraries
# ============================================================
!pip install -q requests tensorflow joblib scikit-learn pandas

import joblib
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 3. Load Model, Scalers, and Selected Features
# ============================================================
model = load_model(model_folder + 'best_aqi_model.h5', compile=False)
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

scaler_X = joblib.load(model_folder + 'scaler_X.pkl')
scaler_y = joblib.load(model_folder + 'scaler_y.pkl')
selected_features = joblib.load(model_folder + 'selected_features.pkl')

print("✅ Model and scalers loaded successfully.")
print(f"Number of features expected: {len(selected_features)}")

# ============================================================
# 4. API Keys (Replace with your actual keys)
# ============================================================
WEATHERAPI_KEY = 'e42bb758faaf4f0a87652208262003'      # from https://www.weatherapi.com
WAQI_TOKEN     = '9dddf2226d2d3f583ee414f44bb386f82b8886ef'          # from https://aqicn.org/data-platform/token/
CITY = 'Delhi, India'                            # to avoid confusion with Delhi, Canada

# ============================================================
# 5. Fetch Weather Data (last 30 days, demo: repeats current)
# ============================================================
def fetch_weather_last_30_days():
    url = f"http://api.weatherapi.com/v1/current.json?key={WEATHERAPI_KEY}&q={CITY}"
    resp = requests.get(url).json()
    if 'error' in resp:
        raise Exception(f"Weather API error: {resp['error']['message']}")
    current = resp['current']
    data = []
    for days_ago in range(30, 0, -1):
        date = (datetime.now() - timedelta(days=days_ago)).date()
        row = {
            'date': date,
            'temperature_2m_max': current['temp_c'],
            'temperature_2m_min': current['temp_c'],
            'apparent_temperature_max': current['feelslike_c'],
            'apparent_temperature_min': current['feelslike_c'],
            'precipitation_sum': current.get('precip_mm', 0) * 24,
            'rain_sum': current.get('precip_mm', 0) * 24,
            'weather_code': current['condition']['code'],
            'wind_speed_10m_max': current['wind_kph'] / 3.6,           # km/h → m/s
            'wind_gusts_10m_max': current.get('gust_kph', current['wind_kph']) / 3.6,
            'wind_direction_10m_dominant': current['wind_degree'],
        }
        data.append(row)
    df = pd.DataFrame(data).set_index('date')
    df.index = pd.to_datetime(df.index)
    return df

# ============================================================
# 6. Fetch Pollution Data (last 30 days, demo: repeats current)
# ============================================================
def fetch_pollution_last_30_days():
    url = f"https://api.waqi.info/feed/{CITY.split(',')[0]}/?token={WAQI_TOKEN}"
    resp = requests.get(url).json()
    if resp['status'] != 'ok':
        raise Exception(f"WAQI API error: {resp.get('data', 'Unknown')}")
    current = resp['data']['iaqi']
    data = []
    for days_ago in range(30, 0, -1):
        date = (datetime.now() - timedelta(days=days_ago)).date()
        row = {
            'date': date,
            'PM2.5': current.get('pm25', {}).get('v', np.nan),
            'PM10':  current.get('pm10', {}).get('v', np.nan),
            'NO2':   current.get('no2', {}).get('v', np.nan),
            'SO2':   current.get('so2', {}).get('v', np.nan),
            'CO':    current.get('co', {}).get('v', np.nan),
            'Ozone': current.get('o3', {}).get('v', np.nan),
        }
        data.append(row)
    df = pd.DataFrame(data).set_index('date')
    df.index = pd.to_datetime(df.index)
    df['AQI'] = resp['data']['aqi']
    return df

# ============================================================
# 7. Feature Engineering (must match training)
# ============================================================
def engineer_features(df):
    """Re‑create all features exactly as in training, with forward fill for NaNs."""
    # Make sure index is datetime
    df.index = pd.to_datetime(df.index)

    # Add metadata columns from date index
    df['Month'] = df.index.month
    df['Year'] = df.index.year
    df['Days'] = df.index.dayofweek + 1          # Monday=1 … Sunday=7
    df['Holidays_Count'] = 0                    # placeholder, update if you have real data

    # Wind direction encoding
    if 'wind_direction_10m_dominant' in df.columns:
        df['wind_dir_sin'] = np.sin(np.radians(df['wind_direction_10m_dominant']))
        df['wind_dir_cos'] = np.cos(np.radians(df['wind_direction_10m_dominant']))

    # Days since rain
    rain_threshold = 0.1
    df['rain_event'] = (df['precipitation_sum'] > rain_threshold).astype(int)
    df['days_since_rain'] = df.groupby(df['rain_event'].cumsum()).cumcount()

    # Weekend flag
    df['is_weekend'] = (df['Days'].isin([6,7])).astype(int)

    # Month dummies
    month_dummies = pd.get_dummies(df['Month'], prefix='month')
    for m in range(1, 13):
        col = f'month_{m}'
        if col not in month_dummies.columns:
            month_dummies[col] = 0
    df = pd.concat([df, month_dummies], axis=1)

    # AQI lags
    for lag in [1, 2, 3, 7]:
        df[f'AQI_lag_{lag}'] = df['AQI'].shift(lag)

    # AQI rolling statistics
    for window in [3, 7, 30]:
        df[f'AQI_roll_mean_{window}'] = df['AQI'].rolling(window).mean()
        df[f'AQI_roll_std_{window}'] = df['AQI'].rolling(window).std()

    # PM2.5 EWMA
    alpha = 0.3
    df['PM25_ewma'] = df['PM2.5'].ewm(alpha=alpha, adjust=False).mean()

    # Rain × wind interaction
    df['rain_wind_interaction'] = df['precipitation_sum'] * df['wind_speed_10m_max']

    # Pollutant ratios
    df['PM25_PM10_ratio'] = df['PM2.5'] / (df['PM10'] + 1e-6)
    df['NO2_SO2_ratio']   = df['NO2'] / (df['SO2'] + 1e-6)
    df['CO_NO2_ratio']    = df['CO'] / (df['NO2'] + 1e-6)

    # Forward fill to replace NaNs at the beginning
    df.ffill(inplace=True)
    df.dropna(inplace=True)
    return df

!pip install openmeteo-requests

import openmeteo_requests
from datetime import date, timedelta

def fetch_weather_last_30_days():
    end_date = date.today()
    start_date = end_date - timedelta(days=30)

    # Open-Meteo client
    om = openmeteo_requests.Client()
    params = {
        "latitude": 28.6139,
        "longitude": 77.2090,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "daily": ["temperature_2m_max", "temperature_2m_min",
                  "apparent_temperature_max", "apparent_temperature_min",
                  "precipitation_sum", "rain_sum",
                  "weather_code", "wind_speed_10m_max", "wind_gusts_10m_max"]
    }
    responses = om.weather_api("https://api.open-meteo.com/v1/forecast", params=params)
    daily = responses[0].Daily()

    # Build DataFrame
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    data = []
    for i, d in enumerate(dates):
        data.append({
            'date': d,
            'temperature_2m_max': daily.Variables(0).Values(i),
            'temperature_2m_min': daily.Variables(1).Values(i),
            'apparent_temperature_max': daily.Variables(2).Values(i),
            'apparent_temperature_min': daily.Variables(3).Values(i),
            'precipitation_sum': daily.Variables(4).Values(i),
            'rain_sum': daily.Variables(5).Values(i),
            'weather_code': daily.Variables(6).Values(i),
            'wind_speed_10m_max': daily.Variables(7).Values(i),
            'wind_gusts_10m_max': daily.Variables(8).Values(i),
        })
    df = pd.DataFrame(data).set_index('date')
    # Wind direction – we'll set a dummy value (not critical)
    df['wind_direction_10m_dominant'] = 0
    return df
# ============================================================
# 9. Run Prediction
# ============================================================
try:
    pred = predict_next_day()
    print(f"\n🌫️  Predicted AQI for tomorrow: {pred:.2f}")
except Exception as e:
    print(f"❌ Error: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model and scalers loaded successfully.
Number of features expected: 49
❌ Error: name 'predict_next_day' is not defined


In [ ]:
def fetch_pollution_last_30_days():
    import requests
    from datetime import date, timedelta

    end_date = date.today()
    start_date = end_date - timedelta(days=30)

    # OpenAQ request (same as before)
    url = "https://api.openaq.org/v3/measurements"
    params = {
        "location": "Delhi",
        "parameter": ["pm25", "pm10", "no2", "so2", "co", "o3"],
        "date_from": start_date.isoformat(),
        "date_to": end_date.isoformat(),
        "limit": 1000
    }
    headers = {"X-API-Key": "YOUR_OPENAQ_KEY"}   # replace with your key

    resp = requests.get(url, params=params, headers=headers).json()
    if 'results' not in resp:
        raise Exception(f"OpenAQ error: {resp}")

    df = pd.DataFrame(resp['results'])
    if df.empty:
        raise Exception("No pollution data returned.")

    # Process pollution data
    df['date'] = pd.to_datetime(df['date']['utc']).dt.date
    df = df.pivot_table(index='date', columns='parameter', values='value', aggfunc='mean')
    all_dates = pd.date_range(start_date, end_date, freq='D')
    df = df.reindex(all_dates, method='ffill')
    df = df.fillna(0)
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'date'}, inplace=True)
    df.set_index('date', inplace=True)

    # Add AQI from historical data (using the previously loaded aqi_series)
    df['AQI'] = aqi_series.reindex(df.index, method='ffill')
    # If any dates are missing (e.g., after 2024), fill with the last known AQI
    df['AQI'].fillna(aqi_series.iloc[-1], inplace=True)

    return df

In [ ]:
test = fetch_pollution_last_30_days()
print(test.head())

Exception: OpenAQ error: {'detail': 'Not Found'}